# 84장 · 실험 비교와 배포 전 회귀 검사

재정렬을 켰더니 검색 점수는 올랐지만 근거가 없는 문의에도 답하기 시작했습니다. 같은 평가 질문에서 새로 생긴 실패와 비용을 함께 비교합니다.

같은 데이터 버전에서 한 요소만 바꿔 품질·지연·토큰·추정 비용을 비교한다. GitHub Actions에 도구 규칙 검사를 연결하고, 개선된 사례와 악화된 사례를 근거로 배포 또는 보류를 결정한다.

## 평균 뒤에 가려진 질문 찾기

In [ ]:
before = {"Q1": True, "Q2": False, "Q3": True}
after = {"Q1": True, "Q2": True, "Q3": False}
for q in before:
    if before[q] != after[q]:
        print(q, before[q], "→", after[q])

Q2 False → True
Q3 True → False


두 설정 모두 세 문제 중 두 문제에 정답을 냅니다. Q2는 고쳤지만 Q3는 새로 틀렸습니다. Q3가 승인 전 확정처럼 반드시 막아야 할 사례라면 평균만 보고 배포할 수 없습니다.

## 같은 시험 사례로 배포 후보의 회귀 판정하기

어떤 후보를 시험 배포할지 결정합니다. 82장의 순위 개선과 83장의 승인 규칙을 모두 만족해야 후보가 통과합니다. 한 번의 요청이라도 중복 저장됐다면 전체 평균이 올라도 보류합니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/evaluation/trajectory.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/evaluation/trajectory.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'case_id': 'july-deadline',
     'as_of': '2026-07-01',
     'ranked': ['A1', 'A2'],
     'observed': 30,
     'expected': 90,
     'gold': 'A2',
     'mode': 'local',
     'editions': [['A1', '2026-06-01', 30], ['A2', '2026-07-01', 90]],
     'approval': {'item_id': 2,
                  'statuses': [200, 409],
                  'version': 1,
                  'done': True,
                  'mode': 'local-version-test',
                  'events': [['read', 2], ['approve', 2], ['write', 2]],
                  'writes': [2]},
     'release_version': 'edition-sort-v2',
     'before': {'recall': 1.0, 'mrr': 0.5},
     'after': {'recall': 1.0, 'mrr': 1.0},
     'candidate': {'citation': 'A2', 'days': 90},
     'candidate_rank': ['A2', 'A1'],
     'candidate_editions': [['A2', '2026-07-01', 90], ['A1', '2026-06-01', 30]],
     'path_ok': True,
     'write_count': 1,
     'events': [('read', 2), ('approve', 2), ('write', 2)]}
    print("복귀 예시", incoming.name)

내 실습 파일 trajectory.json


## 같은 시험 사례로 배포 후보의 회귀 판정하기 실습

이번에는 앞 장에서 남긴 실행 기록에 같은 검사를 적용합니다.

In [ ]:
def gate(record):
    quality = record["candidate"]["days"] == record["expected"] and record["candidate"]["citation"] == record["gold"]
    rank_ok = record["after"]["mrr"] >= record["before"]["mrr"]
    return "시험 배포" if quality and rank_ok and record["path_ok"] and record["write_count"] == 1 else "보류"
decision = gate(previous)
result = {**previous, "gate": decision}
print(result["case_id"], decision)

july-deadline 시험 배포


gate는 같은 질문의 품질과 승인 조건을 개별 검사합니다. 76장의 CI에서 이 함수를 시험한다면 통과 사례와 보류 사례를 함께 고정하세요. 이 출력은 로컬 배포 판단이며 외부 서비스의 배포 이력은 별도로 남깁니다.

### 직접 해 보기

검색 점수와 답은 고정하고 write_count만 2로 바꾸세요. 판정이 보류로 바뀌어야 합니다.

In [ ]:
# 이전 기록을 복사한 뒤 write_count만 바꾸세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>print(gate({**previous, &quot;write_count&quot;: 2}))</code></pre><pre>보류</pre><p>한 건만 저장해야 한다는 조건이 깨져 배포 결정을 보류합니다.</p></details>

### 85장에서 다시 열 파일

방금 계산한 값을 service-lab/evaluation/gate.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "evaluation/gate.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

gate.json


### 직접 해 보기

Q3를 필수 회귀 사례로 정하세요. 후보 실험의 세 요청 지연과 토큰에서 평균 지연, 총 토큰, 예시 단가 기준 추정 비용을 계산하고 배포 여부를 판정하세요.

In [ ]:
latency_ms = [240, 420, 300]
input_tokens, output_tokens = [100, 200, 300], [10, 20, 30]
# 실습용 가정 단가: 입력 100만 토큰당 1달러, 출력 100만 토큰당 4달러

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre style="white-space:pre-wrap;overflow:auto"><code>latency_ms = [240, 420, 300]
input_tokens, output_tokens = [100, 200, 300], [10, 20, 30]
baseline_ms = [200, 300, 250]
baseline_cost = (500 + 4 * 50) / 1_000_000
print(&quot;기준선&quot;, sum(baseline_ms) / len(baseline_ms), &quot;ms&quot;, 550, &quot;tokens&quot;, f&quot;${baseline_cost:.6f}&quot;)
cost = (sum(input_tokens) + 4 * sum(output_tokens)) / 1_000_000
print(sum(latency_ms) / len(latency_ms), &quot;ms&quot;, sum(input_tokens + output_tokens), &quot;tokens&quot;)
print(f&quot;${cost:.6f}&quot;, &quot;배포&quot; if after[&quot;Q3&quot;] else &quot;보류&quot;)</code></pre><pre style="white-space:pre-wrap;overflow:auto"><code>기준선 250.0 ms 550 tokens $0.000700
320.0 ms 660 tokens
$0.000840 보류
</code></pre></details>

## 수업 서비스에 적용하기

LangSmith에서 동일 데이터셋 버전의 두 실험을 열고 한 요소만 바뀌었는지 확인하세요. 각 질문의 점수와 지연, 모델 사용량을 대조하고 실제 모델 단가로 비용을 다시 계산합니다. 76장 GitHub Actions 검사에 83장 승인·중복 저장 규칙을 단위 테스트로 추가하세요. 승인 순서를 깨뜨린 변경이 CI에서 실패하고 복구한 커밋이 통과하는지 두 실행 링크로 확인합니다.

### 제출 전 확인

같은 데이터 버전에서 한 요소만 바꿔 품질·지연·토큰·추정 비용을 비교한다. GitHub Actions에 도구 규칙 검사를 연결하고, 개선된 사례와 악화된 사례를 근거로 배포 또는 보류를 결정한다.

[공식 문서](https://docs.langchain.com/langsmith/compare-experiment-results)